In [ ]:
import os
import pickle
import numpy as np
from tqdm.notebook import tqdm


from tensorflow.keras.applications.vgg16 import VGG16,preprocess_input
from tensorflow.keras.preprocessing.image import load_img,img_to_array
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input,Dense,LSTM,Embedding,Dropout,add
from tensorflow.keras.utils import to_categorical,plot_model


In [ ]:

BASE_DIR='/content/drive/MyDrive/archive (9)'
WORKING_DIR='/content/drive/MyDrive/Working dir'

In [ ]:

model=VGG16()
model=Model(inputs=model.inputs,outputs=model.layers[-2].output)
print(model.summary())

In [ ]:
features={}
directory = os.path.join(BASE_DIR, 'Images')
for img_name in tqdm(os.listdir(directory)):
    img_path = directory+'/'+img_name
    image = load_img(img_path, target_size=(224,224))
    image = img_to_array(image)
    image = image.reshape((1,image.shape[0], image.shape[1], image.shape[2]))
    image = preprocess_input(image)
    feature = model.predict(image, verbose=0)
    image_id = img_name.split('.')[0]
    features[image_id] = feature

In [ ]:
pickle.dump(features, open(os.path.join(WORKING_DIR, 'features.pkl'), 'wb'))


In [ ]:
with open(os.path.join(WORKING_DIR, 'features.pkl'), 'rb') as f:
    features = pickle.load(f)


In [ ]:
with open(os.path.join(BASE_DIR, 'captions.txt'), 'r') as f:
    next(f)
    captions_doc = f.read()


In [ ]:

mapping={}

for line in tqdm(captions_doc.split('\n')):
    tokens = line.split(',')
    if len(line)<2:
        continue
    image_id, caption = tokens[0], tokens[1:]
    image_id = image_id.split('.')[0]
    #convert caption list to string
    caption = " ".join(caption)
    if image_id not in mapping:
        mapping[image_id] = []
    mapping[image_id].append(caption)


In [ ]:

def clean(mapping):
    for key, captions in mapping.items():
        for i in range(len(captions)):
            # take one caption at a time
            caption = captions[i]
            # preprocessing steps
            # convert to lowercase
            caption = caption.lower()
            # delete digits, special chars, etc.,
            caption = caption.replace('[^A-Za-z]', '')
            # delete additional spaces
            caption = caption.replace('\s+', ' ')
            # add start and end tags to the caption
            caption = 'startseq ' + " ".join([word for word in caption.split() if len(word)>1]) + ' endseq'
            captions[i] = caption


In [ ]:
clean(mapping)

In [ ]:
all_captions = []
for key in mapping:
    for caption in mapping[key]:
        all_captions.append(caption)


In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(all_captions)
vocab_size = len(tokenizer.word_index)+1


In [ ]:
max_length = max(len(caption.split()) for caption in all_captions)
max_length


In [ ]:
image_ids = list(mapping.keys())
split = int(len(image_ids)*0.90)
train = image_ids[:split]
test = image_ids[split:]


In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences

def data_generator(data_keys, mapping, features, tokenizer, max_length, vocab_size, batch_size):
    while True:
        X1, X2, y = list(), list(), list()
        n = 0
        for key in data_keys:
            n += 1
            captions = mapping[key]
            for caption in captions:
                # Encode the sequence
                seq = tokenizer.texts_to_sequences([caption])[0]
                # Split the sequence into X, y pairs
                for i in range(1, len(seq)):
                    in_seq, out_seq = seq[:i], seq[i]
                    # Pad input sequence
                    in_seq = pad_sequences([in_seq], maxlen=max_length)[0]
                    # Encode output sequence
                    out_seq = to_categorical([out_seq], num_classes=vocab_size)[0]

                    # Store sequences
                    X1.append(features[key][0])  # Image features
                    X2.append(in_seq)            # Input sequence
                    y.append(out_seq)            # Output word

            # Yield batch when the size matches batch_size
            if n == batch_size:
                yield (
                  {
                    "image_input": np.array(X1),  # Updated key
                    "text_input": np.array(X2)    # Updated key
                   },
                    np.array(y)
                  )

                # Reset for the next batch
                X1, X2, y = list(), list(), list()
                n = 0


In [ ]:
# image feature model
inputs1 = Input(shape=(4096, ))
fe1 = Dropout(0.4)(inputs1)
fe2 = Dense(256, activation='relu')(fe1)
# sequence feature layers
inputs2 = Input(shape=(max_length, ))
se1 = Embedding(vocab_size, 256, mask_zero=True)(inputs2)
se2 = Dropout(0.4)(se1)
se3 = LSTM(256)(se2)

In [ ]:
#decoder model
decoder1 = add([fe2, se3])
decoder2 = Dense(256, activation='relu')(decoder1)
outputs = Dense(vocab_size, activation='softmax')(decoder2)

model = Model(inputs=[inputs1, inputs2], outputs=outputs)
model.compile(loss='categorical_crossentropy', optimizer='adam')

#plot the model
plot_model(model, show_shapes=True)


In [ ]:
def create_tf_dataset(data_keys, mapping, features, tokenizer, max_length, vocab_size, batch_size):
    # Define generator function
    generator = lambda: data_generator(data_keys, mapping, features, tokenizer, max_length, vocab_size, batch_size)

    # Define output_signature for the dataset
    output_signature = (
        {
            "input_1": tf.TensorSpec(shape=(None, features[data_keys[0]][0].shape[0]), dtype=tf.float32),
            "input_2": tf.TensorSpec(shape=(None, max_length), dtype=tf.int32)
        },
        tf.TensorSpec(shape=(None, vocab_size), dtype=tf.float32)
    )

    # Create dataset
    dataset = tf.data.Dataset.from_generator(generator, output_signature=output_signature)
    return dataset


In [ ]:
from tensorflow.keras.layers import Add

def define_model(vocab_size, max_length):
    # Feature extractor model
    inputs1 = Input(shape=(4096,), dtype='float32', name='image_input')  # Name it explicitly
    fe1 = Dropout(0.5)(inputs1)
    fe2 = Dense(256, activation='relu', dtype='float32')(fe1)

    # Sequence model
    inputs2 = Input(shape=(max_length,), dtype='int32', name='text_input')  # Name it explicitly
    se1 = Embedding(input_dim=vocab_size, output_dim=256, mask_zero=True, dtype='float32')(inputs2)
    se2 = Dropout(0.5)(se1)
    se3 = LSTM(256, dtype='float32')(se2)

    # Decoder model
    decoder1 = Add()([fe2, se3])
    decoder2 = Dense(256, activation='relu', dtype='float32')(decoder1)
    outputs = Dense(vocab_size, activation='softmax', dtype='float32')(decoder2)

    # Compile the model
    model = Model(inputs=[inputs1, inputs2], outputs=outputs)
    model.compile(loss='categorical_crossentropy', optimizer='adam')

    # Print model summary
    model.summary()
    return model


In [ ]:
# train the model
from tensorflow.keras.layers import Add

model = define_model(vocab_size, max_length)
# train the model, run epochs manually and save after each epoch
epochs = 5
batch_size=64
steps = len(train)//batch_size
for i in range(epochs):
    generator = data_generator(train, mapping, features, tokenizer, max_length, vocab_size, batch_size)
    model.fit(generator, epochs=1, steps_per_epoch=steps, verbose=1)


In [ ]:
model.save(WORKING_DIR+'/best_model.h5')


In [ ]:
def idx_to_word(integer, tokenizer):
    for word, index in tokenizer.word_index.items():
        if index == integer:
            return word
    return None


In [ ]:
def predict_caption(model, image, tokenizer, max_length):
    in_text = 'startseq'
    #iterate over the max length of sequence
    for i in range(max_length):
        #encode input sequence
        sequence = tokenizer.texts_to_sequences([in_text])[0]
        #pad the sequence
        sequence = pad_sequences([sequence], max_length)
        #predict next word
        yhat = model.predict([image, sequence], verbose=0)
        #  get index with high probability
        yhat = np.argmax(yhat)
        #convert index to word
        word =  idx_to_word(yhat, tokenizer)
        #stop if word not found
        if word is None:
            break
        #append word as input for generating next word
        in_text += " " + word
        #stop if we reach end tag
        if word == 'endseq':
            break
    return in_text

In [ ]:
from nltk.translate.bleu_score import corpus_bleu
actual, predicted = list(), list()
for key in tqdm(test):
    #get actual caption
    captions = mapping[key]
    #predict the caption for image
    y_pred = predict_caption(model, features[key], tokenizer, max_length)
    actual_captions=[caption.split() for caption in captions]
    y_pred = y_pred.split()
    actual.append(actual_captions)
    predicted.append(y_pred)
    #print('Actual: ', actual_captions)
    #print('Predicted: ', y_pred)
print("BLEU-1: %f" % corpus_bleu(actual, predicted, weights=(1.0, 0, 0, 0)))
print("BLEU-2: %f" % corpus_bleu(actual, predicted, weights=(0.5, 0.5, 0, 0)))

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
def generate_caption(image_name):

    #image_name = "1001773457_577c3a7d70.jpg"
    image_id = image_name.split('.')[0]
    img_path = os.path.join(BASE_DIR, "Images", image_name)
    image = Image.open(img_path)
    captions = mapping[image_id]
    print('---------------------Actual---------------------')
    for caption in captions:
        print(caption)

    y_pred = predict_caption(model, features[image_id], tokenizer, max_length)
    print('--------------------Predicted--------------------')
    print(y_pred)
    plt.imshow(image)


In [ ]:
generate_caption("1001773457_577c3a7d70.jpg")

In [ ]:
generate_caption("109738763_90541ef30d.jpg")


In [ ]:
generate_caption("1026685415_0431cbf574.jpg")
